# Simplified No-Optimization CG-Projection Enclosure

This notebook applies the CG-projection enclosure to the 300 free coordinates of `system_303`. It uses symmetric Jacobi scaling, records the CG search directions, and evaluates the projection contraction every 10 iterations.

There is no relaxation term: the projected radius is exactly

$$
d_m^{\mathrm{proj}} = |\Pi_m|d^0.
$$


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from scipy.linalg import qr, solve


data_dir = Path("system_303")

A_full = np.genfromtxt(data_dir / "stiffness_beam_A.dat", delimiter=",")
b_full = np.genfromtxt(data_dir / "forcing_b.dat")
box = np.genfromtxt(data_dir / "B0_wide_bc.dat", delimiter=",")
x_star_full = np.genfromtxt(data_dir / "soln_x.dat")

# Remove the first three boundary-condition coordinates, as in v3.
A = A_full[3:, 3:]
b = b_full[3:]
ell0 = box[3:, 0]
u0 = box[3:, 1]
x_star = x_star_full[3:]

print(f"system dimension: {len(b)}")


system dimension: 300


## Jacobi scaling

With `scale = sqrt(diag(A))`, solve in the coordinates $y=\mathrm{scale}\,x$ and transform the final enclosure back to the original coordinates.


In [23]:
diagonal = np.diag(A)
if np.any(diagonal <= 0):
    raise ValueError("Jacobi scaling requires a positive diagonal.")

scale = np.sqrt(diagonal)
A_hat = A / scale[:, None] / scale[None, :]
b_hat = b / scale
ell_hat = scale * ell0
u_hat = scale * u0
c0 = 0.5 * (ell_hat + u_hat)

max_iterations = 520
contraction_period = 10


## CG and projection contraction

The first function is the standard fixed-iteration CG recurrence. The second implements the algorithm's independent-direction selection. A-normalizing a direction only rescales its column and therefore does not change the projection. The third function directly evaluates $\widehat x_m$, $\Pi_m$, and the interval hull $|\Pi_m|d^0$.


In [24]:
def conjugate_gradient_directions(A, b, x0, iterations):
    """Run exactly `iterations` CG steps and return their search directions."""
    x = np.array(x0, dtype=float, copy=True)
    r = b - A @ x
    p = r.copy()
    directions = []

    for k in range(iterations):
        Ap = A @ p
        rr = float(r @ r)
        denominator = float(p @ Ap)
        if denominator <= 0 or not np.isfinite(denominator):
            raise RuntimeError(f"Invalid CG denominator at iteration {k}.")

        alpha = rr / denominator
        x = x + alpha * p
        r_next = r - alpha * Ap
        directions.append(p.copy())

        if k < iterations - 1:
            beta = float(r_next @ r_next) / rr
            p = r_next + beta * p
        r = r_next

    return x, np.column_stack(directions)


def select_independent_directions(P_all, A, qr_tolerance=1e-10):
    """A-normalize the recorded directions and retain a pivoted-QR basis."""
    AP_all = A @ P_all
    a_norm_squared = np.sum(P_all * AP_all, axis=0)
    valid = np.isfinite(a_norm_squared) & (a_norm_squared > 0)
    if not np.any(valid):
        raise RuntimeError("CG produced no direction with a positive A-norm.")

    Q = P_all[:, valid] / np.sqrt(a_norm_squared[valid])[None, :]
    _, R, pivots = qr(Q, mode="economic", pivoting=True)
    diagonal = np.abs(np.diag(R))
    candidate_rank = int(np.sum(diagonal > qr_tolerance * diagonal[0]))
    if candidate_rank == 0:
        raise RuntimeError("The recorded CG directions are numerically rank zero.")

    # Retain the largest QR basis whose projected Gram solve is stable.
    for rank in range(candidate_rank, 0, -1):
        P = Q[:, pivots[:rank]]
        G = P.T @ A @ P
        if np.linalg.cond(G) <= 1e12:
            return P

    raise RuntimeError("No numerically stable independent direction set was found.")


def contract_projection(A, b, ell0, u0, P_all):
    """Compute the no-relaxation CG-projection enclosure for one CG prefix."""
    c0 = 0.5 * (ell0 + u0)
    d0 = 0.5 * (u0 - ell0)
    P = select_independent_directions(P_all, A)
    G = P.T @ A @ P

    coefficients = solve(G, P.T @ (b - A @ c0), assume_a="sym")
    x_hat = c0 + P @ coefficients

    Ginv_PTA = solve(G, P.T @ A, assume_a="sym")
    Pi = np.eye(len(b)) - P @ Ginv_PTA
    d_projected = np.abs(Pi) @ d0

    ell = np.maximum(ell0, x_hat - d_projected)
    u = np.minimum(u0, x_hat + d_projected)
    if np.any(ell > u):
        raise RuntimeError("The projection produced an inverted interval.")

    return ell, u


def contract_every(A, b, ell0, u0, P_all, period):
    """Evaluate B_m from the same initial box at every `period` directions."""
    iterations = np.arange(0, P_all.shape[1] + 1, period)
    ell_history = [np.array(ell0, copy=True)]
    u_history = [np.array(u0, copy=True)]

    for iteration in iterations[1:]:
        ell, u = contract_projection(A, b, ell0, u0, P_all[:, :iteration])
        ell_history.append(ell)
        u_history.append(u)

    return iterations, np.array(ell_history), np.array(u_history)


## Run the enclosure


In [25]:
x_cg, P_all = conjugate_gradient_directions(
    A=A_hat,
    b=b_hat,
    x0=c0,
    iterations=max_iterations,
)

iteration_history, ell_history_y, u_history_y = contract_every(
    A=A_hat,
    b=b_hat,
    ell0=ell_hat,
    u0=u_hat,
    P_all=P_all,
    period=contraction_period,
)

# Convert all stored enclosures back to the original x coordinates.
ell_history = ell_history_y / scale[None, :]
u_history = u_history_y / scale[None, :]
ell_final = ell_history[-1]
u_final = u_history[-1]

initial_width = u0 - ell0
width_history = u_history - ell_history
with np.errstate(divide="ignore", invalid="ignore"):
    shrinkage_history = 100.0 * (1.0 - width_history / initial_width[None, :])

contains_solution = np.all((ell_final <= x_star) & (x_star <= u_final))
print(f"CG iterations: {max_iterations}")
print(f"contraction calls: {len(iteration_history) - 1}")
print(f"final mean shrinkage: {np.nanmean(shrinkage_history[-1]):.2f}%")
print(f"final enclosure contains x_star: {contains_solution}")


CG iterations: 520
contraction calls: 52
final mean shrinkage: 66.66%
final enclosure contains x_star: True


## Controllable animations

Each animation includes play/pause buttons and a frame slider. Coordinate indices use Python's zero-based convention, so the `1 mod 3` coordinates are `1, 4, 7, ...`.


In [26]:
all_indices = np.arange(len(initial_width))
mod1_indices = all_indices[all_indices % 3 == 1]


def display_shrinkage_animation(indices, title):
    frame_data = shrinkage_history[:, indices]
    finite_data = frame_data[np.isfinite(frame_data)]
    y_max = max(1.0, float(np.max(finite_data)) * 1.05) if finite_data.size else 1.0

    fig, ax = plt.subplots(figsize=(9, 4.5))
    line, = ax.plot(indices, np.nan_to_num(frame_data[0]), linewidth=1)
    label = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(xlim=(indices[0], indices[-1]), ylim=(0, y_max))
    ax.set_xlabel("Coordinate index")
    ax.set_ylabel("Width reduction (% of original width)")
    ax.set_title(title)
    ax.grid(alpha=0.25)

    def update(frame):
        line.set_ydata(np.nan_to_num(frame_data[frame]))
        label.set_text(f"CG iteration {iteration_history[frame]}")
        return line, label

    animation = FuncAnimation(fig, update, frames=len(iteration_history), interval=180)
    display(HTML(animation.to_jshtml(fps=5, default_mode="loop")))
    plt.close(fig)
    return animation


def display_bounds_animation(indices, title):
    lower_data = ell_history[:, indices]
    upper_data = u_history[:, indices]
    y_min = float(np.min(lower_data))
    y_max = float(np.max(upper_data))
    padding = 0.03 * max(y_max - y_min, 1.0)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    lower_line, = ax.plot(indices, lower_data[0], label="lower bound", linewidth=1)
    upper_line, = ax.plot(indices, upper_data[0], label="upper bound", linewidth=1)
    label = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
    ax.set(xlim=(indices[0], indices[-1]), ylim=(y_min - padding, y_max + padding))
    ax.set_xlabel("Coordinate index")
    ax.set_ylabel("Bound value")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend()

    def update(frame):
        lower_line.set_ydata(lower_data[frame])
        upper_line.set_ydata(upper_data[frame])
        label.set_text(f"CG iteration {iteration_history[frame]}")
        return lower_line, upper_line, label

    animation = FuncAnimation(fig, update, frames=len(iteration_history), interval=180)
    display(HTML(animation.to_jshtml(fps=5, default_mode="loop")))
    plt.close(fig)
    return animation


### Shrinkage along all coordinates


In [27]:
all_coordinate_animation = display_shrinkage_animation(
    all_indices,
    "Shrinkage along all coordinates",
)


### Shrinkage along coordinates 1 mod 3


In [28]:
mod1_shrinkage_animation = display_shrinkage_animation(
    mod1_indices,
    "Shrinkage along coordinates 1 mod 3",
)


### Upper and lower bounds for coordinates 1 mod 3


In [29]:
mod1_bounds_animation = display_bounds_animation(
    mod1_indices,
    "Upper and lower bounds for coordinates 1 mod 3",
)
